In [7]:
import numpy as np
import pandas as pd 
import json
import re
import matplotlib.pyplot as plt
import umap
from IPython.display import display
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from solution_utils import build_tables, get_users_text, vectorize_topic_texts, SEED, find_duplicate_groups, read_dialogs, get_groups, calculate_metrics
from sklearn.metrics import homogeneity_score, completeness_score, silhouette_score, adjusted_rand_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components

In [3]:
DATA_PATH = "hw1_train.jsonl"

TRAIN_LABELS_PATH = "train_split.txt"
VAL_LABELS_PATH = "val_split.txt"
TEST_LABELS_PATH = "test_split.txt"

TOPIC_NAMES = {
    0: "Программирование",
    1: "Вопросы про ИИ",
    2: "Перевод и работа с языками",
    3: "Работа с текстом",
    4: "Объяснение и разъяснение",
    5: "Математика, расчеты, решение задач",
    6: "Прочее",
    7: "Нахождение информации",
}


def read_topic_labels_txt(path):
    """Прочитать строки вида 'dialog_id class_id'.

    Возвращает словарь dialog_id -> числовой класс.
    Порядок строк исходного файла сохраняется.
    """
    labels = {}

    lines = Path(path).read_text(
        encoding="utf-8-sig"
    ).splitlines()

    for line_number, line in enumerate(lines, start=1):
        line = line.strip()

        if not line:
            continue

        parts = line.split()

        if len(parts) != 2:
            raise ValueError(
                f"{path}, строка {line_number}: "
                "ожидается 'dialog_id class_id'. "
                f"Получено: {line!r}"
            )

        dialog_id, label_text = parts

        try:
            label = int(label_text)
        except ValueError as exc:
            raise ValueError(
                f"{path}, строка {line_number}: "
                f"класс должен быть целым числом, получено {label_text!r}"
            ) from exc

        if label not in TOPIC_NAMES:
            raise ValueError(
                f"{path}, строка {line_number}: "
                f"неизвестный класс {label}. Допустимы классы 0–7."
            )

        if dialog_id in labels:
            raise ValueError(
                f"{path}: ID {dialog_id} встречается несколько раз."
            )

        labels[dialog_id] = label

    if not labels:
        raise ValueError(f"Файл разметки {path} пуст.")

    return labels


# 1. Один раз загружаем весь исходный датасет.
raw_dialogs = read_dialogs(DATA_PATH)

dialogs_by_id = {
    dialog["dialog_id"]: dialog
    for dialog in raw_dialogs
}

if len(dialogs_by_id) != len(raw_dialogs):
    raise ValueError(
        "В исходном датасете встречаются повторяющиеся dialog_id."
    )


# 2. Загружаем твою разметку.
train_labels = read_topic_labels_txt(TRAIN_LABELS_PATH)
val_labels = read_topic_labels_txt(VAL_LABELS_PATH)
test_labels = read_topic_labels_txt(TEST_LABELS_PATH)


# 3. Проверяем отсутствие одних и тех же ID в разных частях.
split_id_sets = {
    "train": set(train_labels),
    "val": set(val_labels),
    "test": set(test_labels),
}

for left, right in [
    ("train", "val"),
    ("train", "test"),
    ("val", "test"),
]:
    overlap = split_id_sets[left] & split_id_sets[right]

    if overlap:
        raise ValueError(
            f"Пересечение {left} и {right}: "
            f"{sorted(overlap)[:10]}"
        )


def build_topic_split(labels, dialogs_by_id):
    """Сопоставить разметку исходным диалогам.

    X[i] — список ВСЕХ реплик пользователя одного диалога.
    y[i] — класс этого же диалога.
    ids[i] — его dialog_id.
    """
    missing_ids = set(labels) - set(dialogs_by_id)

    if missing_ids:
        raise ValueError(
            "В исходном датасете не найдены ID: "
            f"{sorted(missing_ids)[:10]}"
        )

    ids = list(labels)

    X = [
        [
            turn["text"]
            for turn in dialogs_by_id[dialog_id]["turns"]
            if turn["role"] == "user"
        ]
        for dialog_id in ids
    ]

    y = np.asarray(
        [labels[dialog_id] for dialog_id in ids],
        dtype=np.int64,
    )

    return X, y, ids


# 4. Получаем выборки в формате для твоей функции векторизации.
X_train, y_train, train_ids = build_topic_split(
    train_labels, dialogs_by_id
)

X_val, y_val, val_ids = build_topic_split(
    val_labels, dialogs_by_id
)

X_test, y_test, test_ids = build_topic_split(
    test_labels, dialogs_by_id
)


# 5. Проверяем, что в train есть все классы из val/test.
unseen_classes = (
    set(y_val) | set(y_test)
) - set(y_train)

if unseen_classes:
    raise ValueError(
        "В val/test есть классы, которых нет в train: "
        f"{sorted(unseen_classes)}"
    )

if len(set(y_train)) < 2:
    raise ValueError(
        "Для обучения классификатора в train нужны хотя бы два класса."
    )


print("Всего исходных диалогов:", len(raw_dialogs))
print("Train:", len(X_train), "| y_train:", y_train.shape)
print("Val:  ", len(X_val), "| y_val:  ", y_val.shape)
print("Test: ", len(X_test), "| y_test: ", y_test.shape)

print("\nПример ID:", train_ids[0])
print("Число реплик пользователя:", len(X_train[0]))
print("Класс:", y_train[0], "—", TOPIC_NAMES[y_train[0]])

Всего исходных диалогов: 10000
Train: 933 | y_train: (933,)
Val:   200 | y_val:   (200,)
Test:  200 | y_test:  (200,)

Пример ID: d767a675896
Число реплик пользователя: 6
Класс: 5 — Математика, расчеты, решение задач


In [11]:
# Если функция лежит в solution_utils.py, раскомментируй:
# from solution_utils import vectorize_topic_texts


# ---------------------------------------------------------
# 1. Обучаем TF-IDF ТОЛЬКО на train.
# ---------------------------------------------------------
X_train_tfidf, vectorizer = vectorize_topic_texts(
    X_train,
    min_df=2,
    max_df=0.95,
    max_features=30_000,
    full_normalize=True,
)

# Для val используем тот же словарь и те же IDF.
X_val_tfidf, _ = vectorize_topic_texts(
    X_val,
    vectorizer=vectorizer,
    full_normalize=True,
)

# Преобразовать test можно, но качество на нём пока не смотрим.
X_test_tfidf, _ = vectorize_topic_texts(
    X_test,
    vectorizer=vectorizer,
    full_normalize=True,
)

print("TF-IDF train:", X_train_tfidf.shape)
print("TF-IDF val:  ", X_val_tfidf.shape)
print("TF-IDF test: ", X_test_tfidf.shape)

for name, matrix in [
    ("train", X_train_tfidf),
    ("val", X_val_tfidf),
    ("test", X_test_tfidf),
]:
    empty_count = int((matrix.getnnz(axis=1) == 0).sum())
    print(f"Диалогов без TF-IDF-признаков в {name}: {empty_count}")


# ---------------------------------------------------------
# 2. Обучаем классификатор.
# ---------------------------------------------------------
model = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=3000,
    random_state=42,
)

model.fit(X_train_tfidf, y_train)


# ---------------------------------------------------------
# 3. Предсказываем классы для val.
# ---------------------------------------------------------
y_val_pred = model.predict(X_val_tfidf)

TF-IDF train: (933, 15179)
TF-IDF val:   (200, 15179)
TF-IDF test:  (200, 15179)
Диалогов без TF-IDF-признаков в train: 0
Диалогов без TF-IDF-признаков в val: 0
Диалогов без TF-IDF-признаков в test: 0


In [12]:
# Порядок классов для метрик и матрицы ошибок.
class_ids = list(TOPIC_NAMES)

# Предсказания на validation.
y_val_pred = model.predict(X_val_tfidf)

# Все метрики считаем твоей функцией.
val_metrics = calculate_metrics(
    y_true=y_val,
    y_pred=y_val_pred,
    labels=class_ids,
)


# 1. Общие метрики.
overall_metrics_df = pd.DataFrame(
    {
        "Значение": {
            name: val_metrics[name]
            for name in [
                "accuracy",
                "balanced_accuracy",
                "macro-f1",
                "weighted-f1",
            ]
        }
    }
)

display(overall_metrics_df.style.format("{:.3f}"))


# 2. Precision, recall и F1 по каждому классу.
per_class = val_metrics["per_class_metrics"]

per_class_df = pd.DataFrame(
    [
        {
            "Класс": label,
            "Тема": TOPIC_NAMES[label],
            "Precision": per_class[f"precision_{label}"],
            "Recall": per_class[f"recall_{label}"],
            "F1": per_class[f"f1_{label}"],
        }
        for label in class_ids
    ]
).set_index("Класс")

display(
    per_class_df.style.format(
        {
            "Precision": "{:.3f}",
            "Recall": "{:.3f}",
            "F1": "{:.3f}",
        }
    )
)


# 3. Матрица ошибок.
confusion_df = pd.DataFrame(
    val_metrics["confusion_matrix"],
    index=pd.Index(class_ids, name="Истинный класс"),
    columns=pd.Index(class_ids, name="Предсказанный класс"),
)

print("Строки — истинный класс, столбцы — предсказанный.")
display(confusion_df)


# 4. Ошибочные предсказания для ручного просмотра.
val_errors = pd.DataFrame(
    {
        "dialog_id": val_ids,
        "true_topic": y_val,
        "predicted_topic": y_val_pred,
        "text": ["\n\n".join(parts) for parts in X_val],
    }
)

val_errors = val_errors.loc[
    val_errors["true_topic"] != val_errors["predicted_topic"]
].copy()

display(val_errors.head(10))

,Значение
accuracy,0.630
balanced_accuracy,0.487
macro-f1,0.515
weighted-f1,0.615


,Тема,Precision,Recall,F1
Класс,,,,
0,Программирование,0.875,1.000,0.933
1,Вопросы про ИИ,1.000,0.600,0.750
2,Перевод и работа с языками,0.500,0.385,0.435
3,Работа с текстом,0.689,0.739,0.713
4,Объяснение и разъяснение,0.318,0.452,0.373
5,"Математика, расчеты, решение задач",0.800,0.444,0.571
6,Прочее,0.000,0.000,0.000
7,Нахождение информации,0.438,0.280,0.341


Строки — истинный класс, столбцы — предсказанный.


Предсказанный класс,0,1,2,3,4,5,6,7
Истинный класс,,,,,,,,
0,42,0,0,0,0,0,0,0
1,0,3,1,1,0,0,0,0
2,1,0,5,3,3,0,0,1
3,1,0,2,51,9,0,0,6
4,1,0,2,13,14,0,0,1
5,2,0,0,1,2,4,0,0
6,0,0,0,2,3,0,0,1
7,1,0,0,3,13,1,0,7


,dialog_id,true_topic,predicted_topic,text
0,d46db20e101,6,4,what is the best cast for a live action movie ...
1,d13c1ae2adc,4,3,"What does ""Mälzel"" mean?\n\nWould a civilizati..."
2,dec95de6e11,3,4,To identify the features of our meal kit deliv...
5,df290e1a2ae,7,5,Lighthill-Whitham-Richards (LWR) Model: This m...
11,dd7464cde71,2,4,What is the invoice due date to the transporta...
15,d7c3a317263,3,7,"Who was Дьяків Сава Федерович?\n\nWell, I know..."
22,d3f277383ac,4,2,does stefan in tvd need to sleep ?\n\nwhat is ...
23,df9877b1fa9,1,3,write 20 supervised learning algorithms\n\ntyp...
25,dc6bf43daa5,4,3,Why must businesses embrace breakthrough techn...
27,d1b3ce0893d,7,4,Can a store owner get products in bulk without...
